## Data collecting


Generates the entire dataset from scratch — 10,000  customers and 25,000 transactions, these datasets are not totally

1 Customer 

1.1 Set up libraries

Imports json for saving files, random for generating fake data, and os for handling file paths.

In [ ]:
import json
import random
import os

1.2 Setup path

In [ ]:
OUTPUT_PATH = os.path.join(os.getcwd(), 'customers.json')

1.3 The Generation Function

Generate_customers(output_file, total=10000)

Creates 10,000 bank customers from scratch. For each customer, it randomly assigns:

    A unique account ID (e.g., ACC_00001)

    Date of birth, gender, and city (Hanoi, HCMC, Da Nang, etc.)

    A random monthly salary (between 5M–50M VND)

    A random account balance (between 1M–500M VND)

    A Transaction Count starting at 0 (updated later)

All customers are saved to customers.json.


In [ ]:
def generate_customers(output_file, total=10000):
    unique_ids = [f'ACC_{i:05d}' for i in range(1, total + 1)]

    work_statuses = ['Employed', 'Self-employed', 'Freelancer', 'Unemployed', 'Student', 'Retired']
    locations     = ['Hanoi', 'HCMC', 'Da Nang', 'Hai Phong', 'Can Tho', 'Nha Trang', 'Vung Tau']

    customers = []
    for cid in unique_ids:
        salary  = random.randint(5, 50) * 1_000_000
        balance = random.randint(1, 500) * 1_000_000
        customers.append({
            'Customer ID':       cid,
            'Date of Birth':     f'{random.randint(1965, 2005)}-{random.randint(1, 12):02d}-{random.randint(1, 28):02d}',
            'Gender':            random.choice(['Male', 'Female', 'Other']),
            'Location':          random.choice(locations),
            'Account balance':   balance,
            'Transaction Count': 0,
            'Working Status':    random.choice(work_statuses),
            'Salary (per month)': salary,
        })

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(customers, f, indent=4, ensure_ascii=False)

    print(f'Generated {len(customers):,} customers -> {output_file}')

2 Transaction

2.1 Setup libraries

Imports datetime and timedelta on top of the basics, needed to generate realistic timestamps spread across the year 2025.

In [ ]:
import json
import random
import os
from datetime import datetime, timedelta

2.2 Setup path 

Ensure 'customers.json' exists in this folder before running

Points the script to read from customers.json and save transactions to transaction.json.

In [ ]:
CUSTOMER_INPUT = 'customers.json'
TRANSACTION_OUTPUT = 'transaction.json'

2.3 The Transaction Generator

generate_transactions(customer_file, output_file, n=25000)

This is the most important function, it generates 25,000 transactions with realistic behavior based on the customer's type:

Customer Label-Transaction detail-Locations

    Normal: Supermarket, Electricity Bill, Netflix - Vietnam cities 

    Gambling: Casino top-up, Betting Wallet - Singapore, Macau, Manila

    Loan Sharking:	Quick loan, Urgent Cash Out - Hanoi, HCMC
    

After generating transactions, the function counts how many transactions each customer made and writes that count back into customers.json, keeping both files in sync.




In [ ]:
def generate_transactions(customer_file, output_file, n=25000):
    with open(customer_file, 'r', encoding='utf-8') as f:
        customers = json.load(f)

    # Build a quick lookup: {Customer ID -> Label}
    customer_lookup = {c['Customer ID']: c.get('Label', 'Normal') for c in customers}
    all_ids = list(customer_lookup.keys())
    print(f'Loaded {len(all_ids):,} customers.')

    # Transaction content per label
    content_map = {
        'Normal': {
            'details': ['Supermarket', 'Electricity Bill', 'Monthly Salary', 'Restaurant',
                        'Starbucks', 'Gas Station', 'Netflix Subscription'],
            'locations':    ['Hanoi - VN', 'HCMC - VN', 'Da Nang - VN', 'Can Tho - VN'],
            'amount_range': (20_000, 10_000_000),       
        },
        'Gambling': {
            'details': ['Casino Online Top-up', 'Betting Wallet Deposit', 'Gaming Chip Purchase',
                        'Virtual Slot Funding', 'P2P Game Transfer'],
            'locations':    ['Singapore - SG', 'Macau - CN', 'Manila - PH', 'Cambodia - KH'],
            'amount_range': (100_000, 50_000_000),      
        },
        'Loan Sharking': {
            'details': ['Quick Loan Disbursement', 'Private Finance Support', 'Urgent Cash Out',
                        'P2P Lending Transfer', 'Interest Payment Received'],
            'locations':    ['Hanoi - VN', 'HCMC - VN', 'Hai Phong - VN'],
            'amount_range': (5_000_000, 100_000_000),  
        },
    }

    devices = ['iPhone 15', 'Samsung S23', 'MacBook Air', 'Web Browser', 'Android Phone']

    transactions = []
    for i in range(n):
        sender_id = random.choice(all_ids)
        label     = customer_lookup[sender_id]
        config    = content_map.get(label, content_map['Normal'])

        # Fraudulent transactions tend to happen at odd hours
        hour = random.choice([23, 0, 1, 2, 3, 4, 12, 13]) if label != 'Normal' else random.randint(7, 22)

        date      = datetime(2025, 1, 1) + timedelta(days=random.randint(0, 364))
        timestamp = date.replace(hour=hour, minute=random.randint(0, 59)).strftime('%Y-%m-%d %H:%M:%S')

        transactions.append({
            'Transaction ID':     f'TXN_{300001 + i}',
            'Sender Account ID':  sender_id,
            'Receiver Account ID': f'REC_{random.randint(1000, 9999)}',
            'Transaction amount': random.randint(*config['amount_range']),
            'Timestamp':           timestamp,
            'Transaction Detail': random.choice(config['details']),
            'Geological':          random.choice(config['locations']),
            'Device Use':          random.choice(devices),
        })

    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(transactions, f, indent=4, ensure_ascii=False)

    print(f'Generated {len(transactions):,} transactions -> {output_file}')

    # Update Customer file transaction counts
    counts = {}
    for txn in transactions:
        sid = txn['Sender Account ID']
        counts[sid] = counts.get(sid, 0) + 1

    for c in customers:
        if c['Customer ID'] in counts:
            c['Transaction Count'] = counts[c['Customer ID']]
        else:
             c['Transaction Count'] = 0

    with open(customer_file, 'w', encoding='utf-8') as f:
        json.dump(customers, f, indent=4, ensure_ascii=False)
    
    print(f'Synced Transaction Counts back to -> {customer_file}')